<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom:3px solid rgb(255,106,0); padding-bottom:1em;'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 1: Tensors &amp; Autograd</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn

<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Tensors
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch's fundamental data structure — an $n$-dimensional array, like numpy but with GPU support and automatic differentiation.

Key attribute: `requires_grad` — tells PyTorch to track operations on this tensor so gradients can be computed.

https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.requires_grad.html

In [2]:
# ordinary tensor — no gradient tracking
x = torch.tensor([2.0, 3.0])
print(f'requires_grad: {x.requires_grad}')
print(f'grad_fn:       {x.grad_fn}')


requires_grad: False
grad_fn:       None


In [3]:
# tensor with gradient tracking
theta = torch.tensor([1.0, 0.5], requires_grad=True)
print(f'requires_grad: {theta.requires_grad}')
print(f'grad_fn:       {theta.grad_fn}')   # None — leaf node


requires_grad: True
grad_fn:       None


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; creating leaf tensors with requires_grad
</div>

<!-- FULL: keep, DEMO: delete -->
Three equivalent ways to create a trainable parameter tensor:

- `torch.tensor([...], requires_grad=True)` — from Python data
- `torch.zeros(...).requires_grad_(True)` — in-place flag
- `nn.Parameter(...)` — preferred inside `nn.Module` (covered later)


In [4]:
theta_a = torch.tensor([1.0, 0.5], requires_grad=True)
theta_b = torch.zeros(2).requires_grad_(True)
theta_c = nn.Parameter(torch.tensor([1.0, 0.5]))

print(f'tensor + flag:          requires_grad={theta_a.requires_grad}, grad_fn={theta_a.grad_fn}')
print(f'zeros + requires_grad_: requires_grad={theta_b.requires_grad}, grad_fn={theta_b.grad_fn}')
print(f'parameter:              requires_grad={theta_c.requires_grad}, grad_fn={theta_c.grad_fn}')


tensor + flag:          requires_grad=True, grad_fn=None
zeros + requires_grad_: requires_grad=True, grad_fn=None
parameter:              requires_grad=True, grad_fn=None


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; shared storage — why .clone() matters
</div>

<!-- FULL: keep, DEMO: delete -->
Without `.clone()`, two tensors share the same memory. Modifying one silently changes the other.

https://docs.pytorch.org/docs/2.11/generated/torch.clone.html#torch.clone

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = x          # no copy — shared storage
y[0] = 99.0
print(f'x after modifying y: {x}')   # x also changed!


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])   # reset
z = x.clone()               # independent copy
z[0] = 99.0
print(f'x after modifying z: {x}')   # x unchanged


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Dynamic Computation Graph
</div>

<!-- FULL: keep, DEMO: delete -->
Every operation on a `requires_grad` tensor is recorded in a **dynamic computation graph**. Each result gets a `grad_fn` — a reference to the operation that created it.

The graph has three node types (see Session 4 figure):

- **Input nodes** — data; no gradient tracked
- **Parameter nodes** — learnable; `requires_grad=True`, leaf
- **Compute nodes** — results of operations; have `grad_fn`, not leaf


<!-- FULL: keep, DEMO: delete -->
<img src='../../Common/Pics/CompGraph_1.svg' style='max-width:70%; margin:1em auto; display:block;'/>

In [ ]:
# same decomposition as Session 4
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data


In [26]:
# parameter nodes — requires_grad=True, grad_fn=None (leaf)
print('Parameters:')
for name, t in [('theta_1', theta_1), ('theta_0', theta_0)]:
    print(f'  {name}: requires_grad={t.requires_grad}, grad_fn={t.grad_fn}, is_leaf={t.is_leaf}')


Parameters:
  theta_1: requires_grad=True, grad_fn=None, is_leaf=True
  theta_0: requires_grad=True, grad_fn=None, is_leaf=True


In [27]:
# input nodes — requires_grad=False, grad_fn=None (leaf)
print('Data:')
for name, t in [('X', X), ('y', y)]:
    print(f'  {name}: requires_grad={t.requires_grad}, grad_fn={t.grad_fn}, is_leaf={t.is_leaf}')
print('  -> no gradients computed for data')


Data:
  X: requires_grad=False, grad_fn=None, is_leaf=True
  y: requires_grad=False, grad_fn=None, is_leaf=True
  -> no gradients computed for data


<!-- FULL: keep, DEMO: delete -->
Forward pass builds the graph:

In [30]:
#   (1) yhat = theta_1 * x + theta_0
#   (2) z    = yhat - y
#   (3) L    = z ** 2
yhat = theta_1 * x + theta_0
z    = yhat - y
L    = z**2


In [31]:
# compute nodes — have grad_fn, not leaf
print('Compute nodes:')
for name, t in [('yhat', yhat), ('z', z), ('L', L)]:
    print(f'  {name} {t}, grad_fn={t.grad_fn.__class__.__name__}, is_leaf={t.is_leaf}')


Compute nodes:
  yhat 2.5, grad_fn=AddBackward0, is_leaf=False
  z -0.5, grad_fn=SubBackward0, is_leaf=False
  L 0.25, grad_fn=PowBackward0, is_leaf=False


In [32]:
# each node records what feeds into it
print('L.grad_fn:', L.grad_fn)
print('L.grad_fn.next_functions:')
for fn in L.grad_fn.next_functions:
    print(f'  {fn}')


L.grad_fn: <PowBackward0 object at 0x7eba9736a7d0>
L.grad_fn.next_functions:
  (<SubBackward0 object at 0x7eba7f352620>, 0)


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Detach — removing a tensor from the graph
</div>

<!-- FULL: keep, DEMO: delete -->
`.detach()` cuts a tensor out of the computation graph. The detached tensor has no `grad_fn` — it is treated as a constant.

https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.detach.html#torch.Tensor.detach

No gradient will flow through it. We will see the full consequence after introducing `.backward()`.

In [34]:
print(f'yhat.grad_fn:          {yhat.grad_fn}')   # in graph

yhat_detached = yhat.detach()
print(f'yhat_detached.grad_fn: {yhat_detached.grad_fn}')  # None
print(f'same value:            {torch.allclose(yhat, yhat_detached)}')


yhat.grad_fn:          <AddBackward0 object at 0x7eba9736a7d0>
yhat_detached.grad_fn: None
same value:            True


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>.backward()</code> and <code>.grad</code>
</div>

<!-- FULL: keep, DEMO: delete -->
Calling `.backward()` on a scalar loss traverses the computation graph in reverse, computing $\partial L / \partial \theta$ for every leaf tensor with `requires_grad=True`. The result is stored in `.grad`.

In [ ]:
# fresh setup
theta_1 = torch.tensor(1.0, requires_grad=True)  # parameter
theta_0 = torch.tensor(0.5, requires_grad=True)  # parameter
x = torch.tensor(2.0)  # input data
y = torch.tensor(3.0)  # target data

yhat=2.5, z=-0.5, L=0.25
dyhat/dtheta_1=2.0, dL/dz=-1.0, dL/dtheta1=2zx=-2.0, dL/dtheta0=2z=-1.0


In [45]:
# backward pass
L.backward()


In [46]:
# gradients accumulated at parameter leaves
print(f'theta_1.grad: {theta_1.grad}')   # dL/d(theta_1)
print(f'theta_0.grad: {theta_0.grad}')   # dL/d(theta_0)


theta_1.grad: -2.0
theta_0.grad: -1.0


In [ ]:
# data and compute nodes have no .grad
print(f'X.grad:    {X.grad}')   # None — no requires_grad
print(f'z.grad:    {z.grad}')   # None — compute node, not leaf


In [ ]:
# manual check: dL/d(theta_1) = (2/N) * z.T @ X
N = X.shape[0]
print(f'manual: {(2/N) * z.T @ X}')
print(f'match:  {torch.allclose(theta_1.grad, (2/N) * z.T @ X)}')


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; calling .backward() twice
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch frees the computation graph after `.backward()` to save memory. Calling it again raises a `RuntimeError`. Pass `retain_graph=True` if you need to call it multiple times.

In [ ]:
theta_0 = torch.tensor([[0.2]], requires_grad=True)
L = ((X @ theta_1.T + theta_0 - y) ** 2).mean()
L.backward()   # first call — fine
L.backward()   # second call — RuntimeError!


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Detach revisited — consequence for gradients
</div>

<!-- FULL: keep, DEMO: delete -->
Now that we understand `.backward()`, we can see what detach actually does: gradients do not flow through a detached tensor.

Practical use: **logging the loss** without keeping the graph alive. `.item()` detaches a scalar to a Python float.

In [ ]:
theta_1 = torch.tensor([[1.0, 0.5]], requires_grad=True)
theta_0 = torch.tensor([[0.2]],      requires_grad=True)
yhat = X @ theta_1.T + theta_0
yhat_detached = yhat.detach()


In [ ]:
# backward through original yhat — works
L = ((yhat - y) ** 2).mean()
L.backward()
print(f'theta_1.grad via yhat:          {theta_1.grad}')


In [ ]:
# reset
theta_1 = torch.tensor([[1.0, 0.5]], requires_grad=True)
theta_0 = torch.tensor([[0.2]],      requires_grad=True)
yhat = X @ theta_1.T + theta_0
yhat_d = yhat.detach()

# backward through detached yhat — no gradient flows
L = ((yhat_d - y) ** 2).mean()
L.backward()
print(f'theta_1.grad via yhat_detached: {theta_1.grad}')  # None


<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Manual SGD Training Loop
</div>

<!-- FULL: keep, DEMO: delete -->
The update rule from Session 5: $\theta \leftarrow \theta - \eta \, \nabla_{\theta} L$

Repeat until convergence
1. Forward pass
2. Compute loss
3. Get gradients
4. Update parameters

In [ ]:
# fresh setup
theta_1 = torch.tensor([[1.0, 0.5]], requires_grad=True)
theta_0 = torch.tensor([[0.2]],      requires_grad=True)
X = torch.tensor([[1.0, 2.0], [3.0, 0.5], [2.0, 1.5]])
y = torch.tensor([[3.0], [2.0], [4.0]])

for name, t in [('theta_1', theta_1), ('theta_0', theta_0), ('X', X), ('y', y)]:
    print(f'  {name}: shape={list(t.shape)}, grad_fn={t.grad_fn}, grad={t.grad}, is_leaf={t.is_leaf}, requires_grad={t.requires_grad}')


In [ ]:
# SGD to optimize parameters

lr = 0.01
losses = []  #  montior losses
N = X.shape[0]
for step in range(3):

    # 1. forarward
    yhat = X @ theta_1.T + theta_0

    # 2. loss
    L    = ((yhat - y) ** 2).mean()
    losses.append(L.item())  # https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.item.html#torch.Tensor.item

    print('='*50)
    print(f'step {step} theta_1 before backward: {theta_1}, requires_grad={theta_1.requires_grad}, grad_fn={theta_1.grad_fn}, grad={theta_1.grad}, is_leaf={theta_1.is_leaf}')

    # 3 — backward
    L.backward()

    print(f'step {step} theta_1 after backward: {theta_1}, requires_grad={theta_1.requires_grad}, grad_fn={theta_1.grad_fn}, grad={theta_1.grad}, is_leaf={theta_1.is_leaf}')

    # 4 — update
    # theta_1 = theta_1 - lr * theta_1.grad
    # theta_0 = theta_0 - lr * theta_0.grad

    # 4b — update (must be outside the graph)
    with torch.no_grad():
        theta_1 -= lr * theta_1.grad
        theta_0 -= lr * theta_0.grad
        manual_grad = (2/N) * (yhat-y).T @ X

    print(f'step {step} theta_1 after update: {theta_1}, requires_grad={theta_1.requires_grad}, grad_fn={theta_1.grad_fn}, grad={theta_1.grad}, is_leaf={theta_1.is_leaf}')
    print(f'manual gradient theta_1: {manual_grad}')


print(f'Final loss: {losses[-1]:.4f}')
print(f'theta_1: {theta_1},  theta_0: {theta_0}')

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Why <code>.zero_grad()</code>?
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch **accumulates** gradients — each `.backward()` call **adds** to `.grad`. Always zero gradients before each backward pass in a training loop.

In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)


In [ ]:
# without zeroing — gradients accumulate
for step in range(3):
    loss = (theta[1] * 2.0 + theta[0]) ** 2
    loss.backward()
    print(f'step {step}: theta.grad = {theta.grad}')  # grows!


In [ ]:
# correct — zero before each backward
theta = torch.tensor([1.0, 0.5], requires_grad=True)
for step in range(3):
    if theta.grad is not None: theta.grad.zero_()
    loss = (theta[1] * 2.0 + theta[0]) ** 2
    loss.backward()
    print(f'step {step}: theta.grad = {theta.grad}')


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; zeroing gradients
</div>

<!-- FULL: keep, DEMO: delete -->
- `tensor.grad.zero_()` — zeros in-place
- `tensor.grad = None` — releases memory; slightly faster
- `optimizer.zero_grad()` — preferred in training loops (covered later)
- `optimizer.zero_grad(set_to_none=True)` — default in PyTorch ≥ 2.0


In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
loss  = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()
print(f'grad:           {theta.grad}')


In [ ]:
theta.grad.zero_()
print(f'after zero_():  {theta.grad}')


In [ ]:
# re-run backward to have a gradient, then set to None
loss = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()
theta.grad = None
print(f'after grad=None: {theta.grad}')


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; in-place operations on requires_grad tensors
</div>

<!-- FULL: keep, DEMO: delete -->
In-place operations modify a tensor's data without creating a new graph node. This corrupts the graph PyTorch needs for backpropagation.

In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
z = theta * 2.0
theta += 1.0   # in-place — RuntimeError!


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>torch.no_grad()</code>
</div>

<!-- FULL: keep, DEMO: delete -->
During evaluation — computing validation loss, making predictions — gradients are not needed. `torch.no_grad()` disables graph construction, saving memory and computation.

In [ ]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)


In [ ]:
# with grad tracking (default)
z = theta[1] * 2.0 + theta[0]
print(f'grad tracking on:  z.grad_fn = {z.grad_fn}')


In [ ]:
# without grad tracking
with torch.no_grad():
    z = theta[1] * 2.0 + theta[0]
print(f'grad tracking off: z.grad_fn = {z.grad_fn}')


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Manual SGD Training Loop
</div>

<!-- FULL: keep, DEMO: delete -->
We now have everything needed to train. The update rule from Session 5:

$$\theta \leftarrow \theta - \eta \, \nabla_{\theta} L$$

Implemented directly — no `torch.nn`, no `torch.optim`:
1. Forward pass
2. Compute loss
3. `.backward()` — fills `.grad`
4. Update inside `torch.no_grad()` — update must not be tracked
5. Zero gradients


In [ ]:
# use first feature only for clarity
X_train = X[:, 1:2]

# parameters initialised to zero
theta_1 = torch.zeros(1, 1, requires_grad=True)
theta_0 = torch.zeros(1, 1, requires_grad=True)
lr      = 0.1


In [ ]:
losses = []
for epoch in range(50):

    # 1 & 2 — forward + loss
    yhat = X_train @ theta_1.T + theta_0
    L    = ((yhat - y) ** 2).mean()
    losses.append(L.item())

    # 3 — backward
    L.backward()

    # 4 — update (must be outside the graph)
    with torch.no_grad():
        theta_1 -= lr * theta_1.grad
        theta_0 -= lr * theta_0.grad

    # 5 — zero gradients
    theta_1.grad.zero_()
    theta_0.grad.zero_()

print(f'Final loss: {losses[-1]:.4f}')
print(f'theta_1: {theta_1.item():.4f},  theta_0: {theta_0.item():.4f}')


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses, color='#005564', linewidth=2)
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Training curve — manual SGD', color='#163C69', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


To learn more:

https://docs.pytorch.org/docs/2.11/notes/autograd.html  
https://docs.pytorch.org/docs/2.11/autograd.html


<!-- FULL: keep, DEMO: delete -->
This loop works but is limited:

- **Full-batch** — all data every step
- **Manual parameter management** — error-prone at scale
- **Hard-coded SGD** — rewrite to use Adam, momentum etc.

`nn.Module`, `DataLoader`, and `torch.optim` solve all of these — next notebooks.